# 1.9 DoReMi 数据配比深挖

> 🕐 预估学习时间：40分钟

DoReMi（Domain Reweighting with Minimax Optimization）用代理模型学领域权重，再用于正式预训练，避免手调配比。本节实现核心迭代：proxy 训练 → 领域 excess loss → 更新采样分布。

深挖点：
- 均匀配比 vs 经验配比的失败模式
- Group DRO / minimax 更新
- 代理模型规模与迁移误差
- 与数据退火、课程学习的组合


## 1. 问题设定

K 个领域，采样分布 α。目标不是平均 loss 最低，而是降低**最差领域**的 excess loss（相对参考模型）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

DOMAINS = ['web', 'code', 'math', 'books']
K = len(DOMAINS)


class DomainLM(nn.Module):
    def __init__(self, d=32, vocab=40):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.fc = nn.Linear(d, vocab)

    def forward(self, x):
        return self.fc(self.embed(x).mean(1))


def domain_batch(domain_id, n=64, t=8, vocab=40):
    # each domain has a biased token preference
    x = torch.randint(0, vocab, (n, t))
    x[:, 0] = domain_id * 7 % vocab
    y = (x[:, 0] + domain_id + 1) % vocab
    return x, y


def eval_domain_losses(model, steps=5):
    model.eval()
    losses = []
    with torch.no_grad():
        for k in range(K):
            total = 0.0
            for _ in range(steps):
                x, y = domain_batch(k)
                total += F.cross_entropy(model(x), y).item()
            losses.append(total / steps)
    model.train()
    return torch.tensor(losses)


ref = DomainLM()
# reference: lightly trained on uniform mix
opt = torch.optim.Adam(ref.parameters(), lr=1e-2)
for _ in range(30):
    k = torch.randint(0, K, (1,)).item()
    x, y = domain_batch(k)
    loss = F.cross_entropy(ref(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
ref_losses = eval_domain_losses(ref)
print('=== Reference domain losses ===')
for d, L in zip(DOMAINS, ref_losses.tolist()):
    print(f'{d}: {L:.4f}')
print('Key: Excess loss = proxy_loss(domain) - ref_loss(domain).')


## 2. DoReMi 主循环（教学版）

1. 用当前 α 采样训练 proxy  
2. 估计各领域 excess loss  
3. `α ← normalize(α · exp(η · excess))`（指数梯度 / mirror descent）


In [ ]:
def train_proxy(alpha, steps=40, lr=1e-2):
    model = DomainLM()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for _ in range(steps):
        k = torch.multinomial(alpha, 1).item()
        x, y = domain_batch(k)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(loss.item())
    return model, hist


def doremi(iters=6, eta=1.0):
    alpha = torch.ones(K) / K
    log = []
    for t in range(iters):
        proxy, _ = train_proxy(alpha)
        proxy_losses = eval_domain_losses(proxy)
        excess = (proxy_losses - ref_losses).clamp_min(0)
        # mirror descent update
        alpha = alpha * torch.exp(eta * excess)
        alpha = alpha / alpha.sum()
        log.append((alpha.clone(), excess.clone(), proxy_losses.clone()))
        print(f'iter={t} alpha={[round(a,3) for a in alpha.tolist()]} '
              f'excess={[round(e,3) for e in excess.tolist()]}')
    return alpha, log


print('=== DoReMi Optimization ===')
alpha_star, logs = doremi()
print(f'final alpha={alpha_star.tolist()}')
print('Key: Domains that remain hard get higher sampling weight automatically.')


## 3. 均匀配比对照

若正式训练仍用均匀 α，弱势领域会拖后腿；用 α* 重采样可降低 worst-domain loss。


In [ ]:
def train_main(alpha, steps=80):
    model = DomainLM()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(steps):
        k = torch.multinomial(alpha, 1).item()
        x, y = domain_batch(k)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return eval_domain_losses(model)


uniform = torch.ones(K) / K
L_uni = train_main(uniform)
L_dr = train_main(alpha_star.detach())
print('=== Main-model domain losses ===')
print(f'{"domain":<8} {"uniform":>10} {"doremi":>10}')
for d, a, b in zip(DOMAINS, L_uni.tolist(), L_dr.tolist()):
    print(f'{d:<8} {a:>10.4f} {b:>10.4f}')
print(f'worst uniform={L_uni.max():.4f} doremi={L_dr.max():.4f}')
print('Key: Optimize for worst-domain excess, not only average CE.')


## 4. 工程陷阱

| 陷阱 | 表现 | 缓解 |
|------|------|------|
| proxy 太小 | α* 不可迁移 | proxy ≥ 主模型 1/10 量级试错 |
| 领域定义糊 | 权重抖动 | 清晰 taxonomy + 稳定分类器 |
| 过拟合 excess | 刷某一域 | 温度/截断、正则到先验 α0 |
| 忽略质量 | 垃圾域权重大 | 先质量过滤再 DoReMi |


In [ ]:
# Regularize toward prior alpha0
alpha0 = torch.tensor([0.4, 0.2, 0.2, 0.2])
alpha_reg = 0.7 * alpha_star.detach() + 0.3 * alpha0
alpha_reg = alpha_reg / alpha_reg.sum()
print('=== Prior-regularized alpha ===')
print(list(zip(DOMAINS, alpha_reg.tolist())))
print('Key: Blend learned weights with human prior for production stability.')


## 课后思考题

1. excess loss 相对参考模型，参考模型应如何训练才公平？
2. 多语言 + 多领域同时配比，α 维度爆炸时怎么聚类？
3. DoReMi 与中期训练/数据退火如何衔接？
4. 线上能力偏科时，能否用评测集反向更新 α？风险是什么？

---
> 本节是DoReMi 数据配比的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
